##### STA 220 Data & Web Technologies for Data Analysis

### Lecture 14, 02/19/26, Choropleth maps

#### Announcements



### Today's topics
- Chloropeth maps

## Plotting Spatial Information

Folium can also display boundaries stored in GeoJSON files. See [the documentation](https://python-visualization.github.io/folium/index.html) for more info.

### Chloropleth maps

Folium can also be used to create chloropeth maps. Cloropeth maps are similar to heat maps, in which the units of display are (usually) political entities. They were first introduced in France in the 19th century to color _départements_, which are administrative structures roughly equal in size. 

<div>
    <center>
<img src="https://upload.wikimedia.org/wikipedia/commons/3/38/Carte_figurative_de_l%27instruction_populaire_de_la_France.jpg" width="500"/>
</center>
    </div>

The preceding example about the proportion of literate population is a textbook example of chloropeth maps for unclassed data: The gradient ranges from low to high. 

In [ ]:
import pandas as pd

In [ ]:
# import pandas as pd

eco_footprints = pd.read_csv("../data/footprint.csv")

political_countries_url = (
    "http://geojson.xyz/naturalearth-3.3.0/ne_50m_admin_0_countries.geojson"
)

eco_footprints.head()

In [ ]:
max_eco_footprint = eco_footprints["Ecological footprint"].max()
print(max_eco_footprint)

`state_geo` is a [GeoJSON](https://geojson.org/) file. GeoJSONs identify a region by specifying the nodes of their corresponding polygon as latitude-longitude pairs. We are going to see later how these are structured. 

To understand how we can combine the country with the geojson, have a look at the [political_countries_url](https://geojson.xyz/naturalearth-3.3.0/ne_50m_admin_0_countries.geojson).

In [ ]:
import folium
import folium.plugins

In [ ]:
m = folium.Map(location=(30, 10), zoom_start=3, tiles="cartodb positron")
folium.Choropleth(
    geo_data=political_countries_url,
    data=eco_footprints,
    columns=("Country/region", "Ecological footprint"),
    key_on="feature.properties.name", # keys to link the data with gejson
    bins=(0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, max_eco_footprint), # define the bins manually
    fill_color="RdYlGn_r",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white", # nans shall be left white
    legend_name="Ecological footprint per capita",
    name="Countries by ecological footprint per capita",
).add_to(m)
folium.LayerControl().add_to(m)

folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)

fig = folium.Figure(width = 1000, height = 800)
fig.add_child(m)

m

In [ ]:
fig.save("../output/eco_footprint.html")

In [ ]:
import folium.plugins
m = folium.plugins.DualMap(location=(52.1, 5.1), tiles=None, zoom_start=2)

folium.TileLayer("openstreetmap").add_to(m.m1)
folium.TileLayer("cartodbpositron").add_to(m.m2)

folium.Choropleth(
    geo_data=political_countries_url,
    data=eco_footprints,
    columns=("Country/region", "Ecological footprint"),
    key_on="feature.properties.name",
    bins=(0, 1, 1.5, 2, 3, 4, 5, 6, 7, 8, max_eco_footprint),
    fill_color="RdYlGn_r",
    fill_opacity=0.8,
    line_opacity=0.3,
    nan_fill_color="white",
    legend_name="Ecological footprint per capita",
    name="Countries by ecological footprint per capita",
).add_to(m.m1)

folium.LayerControl().add_to(m.m1)
m

Another example would be to visualize the unemployment rates in the different states of the US.

In [ ]:
# import pandas as pd
import requests

In [ ]:
state_geo = requests.get(
    "https://raw.githubusercontent.com/python-visualization/folium-example-data/main/us_states.json"
).json()
state_data = pd.read_csv(
    "https://raw.githubusercontent.com/python-visualization/folium-example-data/main/us_unemployment_oct_2012.csv"
)

In [ ]:
state_geo

In [ ]:
type(state_geo)

In [ ]:
len(state_geo['features'])

In [ ]:
state_geo['features'][0]

In [ ]:
state_geo['features'][0]['id']

In [ ]:
type(state_data)

In [ ]:
state_data.head()

In [ ]:
state_data.shape

In [ ]:
m = folium.Map(location=[48, -102], zoom_start=3)

folium.Choropleth(
    geo_data=state_geo, # takes GeoJSON
    name="choropleth",
    data=state_data,
    columns=["State", "Unemployment"],
    key_on="feature.id",
    fill_color="YlGn",
    fill_opacity=0.4,
    line_opacity=0.2,
    control_scale=True,
    legend_name="Unemployment Rate (%)",
).add_to(m)

folium.LayerControl().add_to(m)
fig = folium.Figure(width = 800, height = 500)
fig.add_child(m)

In [ ]:
fig.save("../output/us_unemployment.html")

### Election results

Classed maps color political entities by categorical features. The following example shows the party of each winner of constituencies for the 2019 United Kingdom election. 

<div>
    <center>
<img src="https://upload.wikimedia.org/wikipedia/commons/e/e2/2019UKElectionMap.svg" width="400"/>
</center>
</div>

The problem with this map is that a) lots of additional information (e.g., how well did the Conserative party (blue)) do in scotland) is missing and b) Even though the british electoral system is _winner takes all_, the constituencies are displayed in different sizes (e.g., the larger (rural) constituencies overinflate the success of the Conservative party). 

Issue b) is alleviated by variants of this kind: 

<div>
    <center>
<img src="https://miro.medium.com/v2/resize:fit:1400/1*hfA55y_xlYTs5v3k-_AxCA.png" width="200"/>
</center>
</div>

This is one example of preferring regular shapes over accurate constituency boundaries. The size of the constituencies are equal, as each correspond to one seat in parliament. They convey a more truthful message on the election results than constituencies that scale with area. 

As for issue a), classed cloropeth maps are generally unsuitable to display both range and class. However, one could try: 

<div>
<img src="https://upload.wikimedia.org/wikipedia/commons/a/a6/2019_General_Election_Results.png" width="350"/>
</div>

We can scrape the election results from wikipedia. Some data processing is in order. 

Find the results [here](https://en.wikipedia.org/wiki/Results_of_the_2019_United_Kingdom_general_election).

#### Data preprocessing

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}
elections = pd.read_html('https://en.wikipedia.org/wiki/Results_of_the_2024_United_Kingdom_general_election', storage_options = headers) 

In [ ]:
type(elections)

In [ ]:
len(elections)

In [ ]:
elections.pop(0)

In [ ]:
england = elections[0]

In [ ]:
england.head()

In [ ]:
england = england.drop(england.columns[1:6], axis=1)

In [ ]:
england.columns.to_flat_index()[0]

In [ ]:
england

In [ ]:
england = elections[0]
england = england.drop(england.columns[1:6], axis=1)
england.columns = [i[1] for i in england.columns.to_flat_index()]
england = england.rename(columns = {'Party': 'Winner','Lab.[b]': 'Lab'})

In [ ]:
england

In [ ]:
england = england.loc[england.Constituency.notna() 
                          & (england.Constituency != 'All constituencies')]

In [ ]:
england

In [ ]:
england = england[['Constituency', 'Winner', 'Con.', 'Lab', 'Lib. Dems', 'Green', 'Total']]
england.head()

In [ ]:
elections[1].head()

In [ ]:
scotland = elections[1]
scotland = scotland.drop(scotland.columns[1:4], axis=1)
scotland.columns = [i[1] for i in scotland.columns.to_flat_index()]
scotland = scotland.rename(columns = {'Party': 'Winner','Lab.': 'Lab'})
scotland = scotland.loc[scotland.Constituency.notna() 
                          & (scotland.Constituency != 'All constituencies')]
scotland = scotland[['Constituency', 'Winner', 'Con.', 'Lab', 'Lib. Dems', 'Green', 'Total']]
scotland.head() 

In [ ]:
scotland

In [ ]:
wales = elections[2]
wales = wales.drop(wales.columns[1:4], axis=1)
wales.columns = [i[1] for i in wales.columns.to_flat_index()]
wales.head()

In [ ]:
wales = wales.rename(columns = {'Affiliate': 'Winner','Lab.': 'Lab'})
wales = wales.loc[wales.Constituency.notna() 
                          & (wales.Constituency != 'All constituencies')]
wales = wales[['Constituency', 'Winner', 'Con.', 'Lab', 'Lib. Dems', 'Green', 'Total']]
wales.head()

In [ ]:
election = pd.concat([england, scotland, wales]).set_index('Constituency').fillna(0)

winner = election['Winner']
election = election.drop(['Winner'], axis = 1)

In [ ]:
election = pd.concat([england, scotland, wales]).set_index('Constituency').fillna(0)

winner = election['Winner']
election = election.drop(['Winner'], axis = 1)
election = election.replace(to_replace='—N/a', value=0)
for col in election.columns:
    election[col] = election[col].astype(int) / election['Total'].astype(int) # percentage of votes for this party col
election = election.drop('Total', axis = 1)

election.head()

In [ ]:
winner.head()

#### GeoJSON Files

The geographical information on the constituencies is available as GeoJSON online. For GeoJSON see [here](https://geojson.org/) and [here](https://en.wikipedia.org/wiki/GeoJSON). We now have to merge both files. 

In [ ]:
import folium

m = folium.Map(width = 800, height = 600, location=[50, 0], zoom_start=4)

folium.Choropleth(
    geo_data='../data/geoConstituencies.json', # takes GeoJSON
).add_to(m)

# show_map(m, w = 800, h =600)
m

In [ ]:
import json

with open('../data/geoConstituencies.json', 'r') as file:
    boundaries = json.load(file)

In [ ]:
boundaries["features"][0] # first Constituency. 

Since most libraries are only able to retrieve information under first-level node, we have to modify the GeoJSON to make the names easily accessible. 

In [ ]:
for feature in boundaries['features']:
    feature['PCON24NM'] = feature['properties']['PCON24NM']

In [ ]:
json_const = [b['PCON24NM'] for b in boundaries["features"]]

In [ ]:
json_const

Some constituencies in our election data have non-unicode names. They will not be matched correctly.

In [ ]:
election.index

In [ ]:
import re
from unidecode import unidecode

standardize = lambda x: unidecode(re.sub(',', '', x))
election.index = [standardize(i) for i in election.index]

election.index

In [ ]:
election.index[518] # given as Weston-Super-Mare in boundaries! 

In [ ]:
# election = election.rename(index = {'Weston-super-Mare': 'Weston-Super-Mare'})

In [ ]:
election.shape

In [ ]:
election.head()

In [ ]:
election_sorted = election.sort_index()

In [ ]:
json_const

In [ ]:
[a for a in election_sorted.index if a not in json_const]

In [ ]:
json_const[0] = 'Ynys Mon'

In [ ]:
election_sorted

Any remaining mismatches of the data and GeoJSON file that contains the polygons will have to be dealt with later.  

We want to color the map according to how good each party did in the constituency. 

In [ ]:
election = dict(election)
election['Green']['Aldershot'] 

Lets assign each party a color. `branca.colormap.LinearColormap` create a linar interpolation between two colors. 

In [ ]:
!pip install cmp

In [ ]:
import branca.colormap as cmp
import numpy as np

colors = {'fire': cmp.LinearColormap(['white', color], vmin=0, vmax=np.round(max(1.01*election[party]),1)) \
          for party, color in zip(election.keys(), ['#3a85d6', '#ed4224', '#e8ca54', '#6cbd6c'])}

In [ ]:
colors = {party: cmp.LinearColormap(['white', color], vmin=0, vmax=np.round(max(1.01*election[party]),1)) \
          for party, color in zip(election.keys(), ['#3a85d6', '#ed4224', '#e8ca54', '#6cbd6c'])}

In [ ]:
colors['Con.']

In [ ]:
election['Con.']['Aldershot']

In [ ]:
colors['Con.'](0.5)

The custom coloring `get_color` takes the constituency name from the GeoJSON, removes commas (to deal with another mismatch: 'Birmingham, Edgbaston' to 'Birmingham Edgbaston') and, if data is available for that polygon, colors it according to the vote share.  

In [ ]:
def get_color(feature, party):
    value = feature['PCON24NM']
    value = re.sub(',', '', value)
    
    #print(value)
    
    return colors[party](election[party].get(value,0))

In [ ]:
boundaries['features'][0]

In [ ]:
get_color(boundaries['features'][0], 'Lab') # [0] for the first constituency

Lets create a map. We set `tiles` to `False` to remove the standard openstreetview map. 

We can use some default [tiles](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html) or use custom ones. 

I chose to use one of [Esri](https://server.arcgisonline.com/arcgis/rest/services)s open maps with a monochrome background. 

In [ ]:
import folium 
m = folium.Map(location=[54.9, -4], zoom_start=6, 
               width=600, height=750, 
               tiles = None)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite', overlay=True, control=False
).add_to(m)
fig = folium.Figure(width = 600, height = 750)
fig.add_child(m)
folium.plugins.Fullscreen().add_to(m)
m

In [ ]:
boundaries['features'][0]

Now, I would like to add each party's performance onto the map sequentially.  Note that we pass `get_color` to the `style_function` argument. The additional parameters govern the boundaries, opacity, and `overlay=False` ensures that each object is given a radio buttion, not a checkmark button. 

In [ ]:
get_color

In [ ]:
for i in ['Con.', 'Lab', 'Lib. Dems', 'Green']: 
    fg = folium.FeatureGroup(name=i, overlay=False)

    folium.GeoJson(
            boundaries,
            style_function=lambda feature, party=i: {
                "fillColor": get_color(feature, party),
                "color": "gray",
                "weight": 1,
                "dashArray": "1",
                "fillOpacity": 1,
            }, popup=folium.GeoJsonPopup(fields=["PCON24NM"], aliases = ['Constituency'])
        ).add_to(fg)
    
    fg.add_to(m)
    

folium.LayerControl(collapsed=False).add_to(m)

m

Even though this map does not use regular shapes do map each constituency, we learn, e.g., that the Tories do better in rural areas, while Labour underperformes in these. With notable exceptions, the LibDems are stronger in the rural south.

In [ ]:
m.save("../output/british2024_election.html")

In [ ]:
!open ../output/british2024_election.html

Even though this map does not use regular shapes do map each constituency, we learn, e.g., that the Tories do better in rural areas, while Labour underperformes in these. With notable exceptions, the LibDems are stronger in the rural south. 

While gradual color schemes are most appropriate for chloropeth maps, they only allow to show a single feature. 

Another problem in chloropeth maps is that they do not accurately depict data over geographic space with the use of large blocks. 

Dasyncretic maps address this issue. They use auxiliary information to portray the data more accurately. They intersect geographical objects to filter out spatial information that does not contribute to the data. 

<div>
    <center>
<img src="https://upload.wikimedia.org/wikipedia/commons/7/7e/Utah_Valley_dasymetric_map.png" width="1000"/>
</center>
</div>

### Dot maps

Another popular map format are dot maps. Consider the following map from the 1931 Polish census. 

<div>
    <center>
<img src="https://upload.wikimedia.org/wikipedia/commons/2/25/GUS_languages1931_Poland.jpg", width = "600" />
        </center>
</div>

Lets give this map a modern touch! We will draw from [Paul Dziemielas](https://dziemiela.com/personal/interwar_poland.html) geographical boundaries and census results. 

In [ ]:
import requests
r = requests.get('https://www.dziemiela.com/personal/Interwar_Poland_1934_20142.json', headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
})
topoJSON = r.json() # this is in topoJSON format!

In [ ]:
topoJSON['objects']['Palatinates']['geometries']

In [ ]:
import folium
m = folium.Map(width=800, height=700, tiles = None,
               location=[52, 23], zoom_start=6)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(m)

In [ ]:
#topoJSON['objects']['Districts']['geometries']#[0]['properties']['GEOID']

In [ ]:
folium.TopoJson(topoJSON,
    name = "Districts",
    object_path='objects.Districts', 
    style_function=lambda feature: {
        "fillColor": None,
        "fillOpacity": 0.0,
        "color": "lightgray",
        "weight": 1,
        "dashArray": "1",
    }, overlay=True, control=True).add_to(m)

In [ ]:
fig = folium.Figure(width=800, height=700)
fig.add_child(m)

folium.LayerControl().add_to(m)
m

Lets retrieve the census data from the same source.

In [ ]:
import requests, zipfile, io

r = requests.get('https://www.dziemiela.com/personal/Interwar_Poland_1934_20142.zip', headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
})
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("../data/polish_census")

`fiona` is a module to handle geopackages. We have data for the 1931 and 1921 census, and a school census of 1926. We are only interested in the 1931 census. 

In [ ]:
!pip install fiona

In [ ]:
import fiona
fiona.listlayers('../data/polish_census/Interwar_Poland_1934.gpkg')

In [ ]:
import geopandas
districts = geopandas.read_file("../data/polish_census/Interwar_Poland_1934.gpkg", 
                                layer='Census_1931_Districts') 
districts.head(3)

In [ ]:
print('\n'.join(districts.columns))

Lets craft the data set that is used to plot dots. 

In [ ]:
import numpy as np
import pandas as pd
data = districts[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                    'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna()
#data.head()

In [ ]:
data = data.apply(lambda x: np.floor(x / 10000).astype(int), axis = 1)

In [ ]:
data.head()

As for the UK election, choose colors for each category. 

In [ ]:
colorpicker = {lang: color for lang, color in zip(data.columns, 
    ['#de3e16', '#f7d914', '#1cbd87', '#36a334', '#b569e0', '#64a8ed', '#b9d676', '#f781b2'])}

In [ ]:
import matplotlib.pyplot as plt

y = [0, 1]
x = [1, 1]

fig, axes = plt.subplots(ncols=4,nrows=2, sharex=True, sharey=True,
                         figsize=(5,2), subplot_kw={'xticks': [], 'yticks': []})

for ax, key in zip(axes.flat, colorpicker.keys()):
    ax.plot(x, y)
    ax.fill_betweenx(y, 0, 1, facecolor=colorpicker[key])
    ax.set_xlim(0, 0.1)
    ax.set_ylim(0, 1)
    ax.set_title(str(key))

plt.tight_layout()
plt.show()

Even though topoJSON is a more economical data format, we want to generate random points in each geometric object. To do so, we need to recast the topoJSON in to geoJSON format. 

In [ ]:
!pip install pytopojson

In [ ]:
from pytopojson import feature
feature_ = feature.Feature()
geojson = feature_(topoJSON, 'Districts')

In [ ]:
geojson['features'][0] # navigate through... / do not print

In [ ]:
gdf = geopandas.GeoDataFrame.from_features(geojson['features'])
gdf.head(2)

In [ ]:
gdf['geometry'][2].bounds

In [ ]:
gdf['geometry'][2]

Random (on the cartesian plane) points are generated in each object. 

In [ ]:
import shapely, random
def generate_random_points(number, GEOID):

    # Select list entry of given object
    polygon = gdf[gdf['GEOID'] == GEOID]['geometry']#[0]
    # Extract bounding box (extent) from the GeoDataFrame
    minx, miny, maxx, maxy = polygon.bounds.squeeze()
    
    # Generate random points within the bounding box
    random_points = []
    while len(random_points) < number:
        random_point = shapely.geometry.Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        # Check if the point is inside any of the polygons
        if all(random_point.intersects(polygon)):
            random_points.append(random_point)

    return geopandas.GeoDataFrame(geometry=random_points)['geometry']

In [ ]:
generate_random_points(2, 'P1613')

In [ ]:
generate_random_points(1, 'P1613')[0]

Finally, lets add the dots to the map. 

In [ ]:
m = folium.Map(width=700, height=700, tiles = None,
               location=[52, 23], zoom_start=6)
tile = folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(m)

folium.TopoJson(topoJSON,
    object_path='objects.Districts', 
    style_function=lambda feature: {
        "fillColor": None,
        "fillOpacity": 0.0,
        "color": "lightgray",
        "weight": 1,
        "dashArray": "1",
    }, overlay=False, control=False).add_to(m)

for lang, countsvector in dict(data).items():
    color = colorpicker[lang]
    fg = folium.FeatureGroup(name=lang).add_to(m)
    for GEOID, counts in dict(countsvector).items(): 
        for point in generate_random_points(counts, GEOID): 
            folium.CircleMarker(location=[point.y, point.x], 
                    stroke=False,
                    fill=True,
                    color=color, 
                    fill_opacity=1,
                    radius=2).add_to(fg)

In [ ]:
fig = folium.Figure(width = 700, height = 700)
fig.add_child(m)

folium.LayerControl(position='bottomleft', collapsed = False).add_to(m)
m 

<div>
    <center>
<img src="https://upload.wikimedia.org/wikipedia/commons/2/25/GUS_languages1931_Poland.jpg" width="1000"/>
</center>
    </div>

So why did the Polish census agency decide for a dot map? Lets create a plurality map. 

In [ ]:
districts[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                            'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna()

In [ ]:
district_colors = districts[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                            'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna().idxmax(axis=1)
district_colors

In [ ]:
colorpicker

Lets add the palatinates as well. 

In [ ]:
palatinates = geopandas.read_file("../data/polish_census/Interwar_Poland_1934.gpkg", layer='Census_1931_Palatinates')
palatinates.head()

In [ ]:
palatinates[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                                 'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna()

In [ ]:
palatinate_colors = palatinates[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                                 'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna().idxmax(axis=1)

In [ ]:
palatinate_colors

In [ ]:
m = folium.Map(width=800, height=700, tiles = None,
               location=[52, 23], zoom_start=6)
base_map = folium.FeatureGroup(name='Basemap', overlay=False, control=False)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(base_map)
base_map.add_to(m)

folium.TopoJson(topoJSON,
    name = 'Palatinates',
    object_path='objects.Palatinates', 
    style_function=lambda feature: {
        "fillColor": colorpicker[palatinate_colors[feature['properties']['GEOID']]],
        "fillOpacity": 0.8,
        "color": "lightgray",
        "weight": 1,
        "dashArray": "1",
    }, overlay=False).add_to(m)


folium.TopoJson(topoJSON,
    name = "Districts",
    object_path='objects.Districts', 
    style_function=lambda feature: {
        "fillColor": colorpicker[district_colors[feature['properties']['GEOID']]],
        "fillOpacity": 0.8,
        "color": "lightgray",
        "weight": 1,
        "dashArray": "1",
    }, overlay=False).add_to(m)

In [ ]:
fig = folium.Figure(width = 800, height = 700)
fig.add_child(m)
folium.LayerControl(collapsed = False).add_to(m)
m

The actual map from the census did only consider the categories 'Polish' or 'Other'. 

In [ ]:
district_colors = districts[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                            'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna()

district_colors = pd.DataFrame({"POLISH": district_colors['POLISH'], 
                                "OTHER": district_colors.drop('POLISH', axis=1).sum(axis=1)}).idxmax(axis=1)

In [ ]:
palatinates = geopandas.read_file("../data/polish_census/Interwar_Poland_1934.gpkg", layer='Census_1931_Palatinates')
palatinate_colors = palatinates[['GEOID', 'POLISH', 'UKRAINIAN', 'RUSKI', 
                                 'BELARUSIAN', 'LITHUANIAN', 'GERMAN', 'YIDDISH', 'HEBREW']].set_index('GEOID').dropna()

palatinate_colors = pd.DataFrame({"POLISH": palatinate_colors['POLISH'], 
                                  "OTHER": palatinate_colors.drop('POLISH', axis=1).sum(axis=1)}).idxmax(axis=1)

In [ ]:
colorpicker["OTHER"] = '#b9d676'

In [ ]:
m = folium.Map(width=800, height=800, tiles = None,
               location=[53, 23], zoom_start=5)
base_map = folium.FeatureGroup(name='Basemap', overlay=True, control=False)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(base_map)
base_map.add_to(m)

folium.TopoJson(topoJSON,
    name = "Districts",
    object_path='objects.Districts', 
    style_function=lambda feature: {
        "fillColor": colorpicker[district_colors[feature['properties']['GEOID']]],
        "fillOpacity": 0.8,
        "color": "lightgray",
        "weight": 1,
        "dashArray": "1",
    }, overlay=False).add_to(m)

folium.TopoJson(topoJSON,
    name = 'Palatinates',
    object_path='objects.Palatinates', 
    style_function=lambda feature: {
        "fillColor": colorpicker[palatinate_colors[feature['properties']['GEOID']]],
        "fillOpacity": 0.8,
        "color": "lightgray",
        "weight": 1,
        "dashArray": "1",
    }, overlay=False).add_to(m)

In [ ]:
folium.LayerControl(collapsed = False).add_to(m)
m 